# Multi-Modal Neural Network for Construction Cost Prediction

This notebook builds a model combining:
- Tabular data (MLP)
- Sentinel-2 imagery (CNN)
- VIIRS imagery (CNN)

We fuse all modalities into a final regression model.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from pathlib import Path

DataPath = Path("..") / "Processed data"

train_df = pd.read_csv(DataPath / "processed_data.csv")

print(train_df.shape)
train_df.head()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Tabular Model (MLP with Embeddings)

In [ ]:
class TabularModel(nn.Module):
    def __init__(self, num_numeric, cat_dims, emb_dims):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_dim, emb_dim)
            for cat_dim, emb_dim in zip(cat_dims, emb_dims)
        ])

        emb_total_dim = sum(emb_dims)

        self.mlp = nn.Sequential(
            nn.Linear(num_numeric + emb_total_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

    def forward(self, x_numeric, x_categorical):
        emb = [emb_layer(x_categorical[:, i]) 
               for i, emb_layer in enumerate(self.embeddings)]
        emb = torch.cat(emb, dim=1)

        x = torch.cat([x_numeric, emb], dim=1)
        return self.mlp(x)


## CNN Image Encoder (ResNet Backbone)

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, output_dim=128):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, output_dim)

    def forward(self, x):
        return self.backbone(x)


## Fusion Model

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, tabular_model, sentinel_model, viirs_model):
        super().__init__()

        self.tabular = tabular_model
        self.sentinel = sentinel_model
        self.viirs = viirs_model

        self.head = nn.Sequential(
            nn.Linear(64 + 128 + 128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x_num, x_cat, img_sentinel, img_viirs):
        t_feat = self.tabular(x_num, x_cat)
        s_feat = self.sentinel(img_sentinel)
        v_feat = self.viirs(img_viirs)

        x = torch.cat([t_feat, s_feat, v_feat], dim=1)
        return self.head(x)


## Loss Function

In [ ]:
criterion = nn.SmoothL1Loss()

## Example Model Initialization

In [ ]:
num_numeric = 10  # adjust
cat_cols = [
    "country",
    "region_economic_classification",
    "seismic_hazard_zone",
    "tropical_cyclone_wind_risk",
    "tornadoes_wind_risk",
    "koppen_climate_zone",
    "geolocation_name"
]

cat_dims = [train_df[col].nunique() for col in cat_cols]
emb_dims = [min(50, (dim + 1) // 2) for dim in cat_dims]

tabular_model = TabularModel(num_numeric, cat_dims, emb_dims)
sentinel_model = ImageEncoder(128)
viirs_model = ImageEncoder(128)

model = FusionModel(tabular_model, sentinel_model, viirs_model)

model.to(device)


## Training Step (Skeleton)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_step(batch):
    model.train()

    x_num = batch["numeric"].to(device)
    x_cat = batch["categorical"].to(device)
    img_s = batch["sentinel_img"].to(device)
    img_v = batch["viirs_img"].to(device)
    y = batch["target"].to(device)

    preds = model(x_num, x_cat, img_s, img_v)

    loss = criterion(preds.squeeze(), y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()
